# 歌词分词，词性标注

In [1]:

import json
import pandas as pd

import jieba
import jieba.posseg as pseg
from collections import Counter
from openai import OpenAI

In [2]:
# 可以选择是否加载
# jieba.load_userdict('data/mayday_dict_simple.txt')

In [3]:
import sys
sys.path.append('..')

# 歌曲数据

In [4]:
songs_file_path = 'data_output/liuyuning/qq_songs_data_ost.csv'
df_songs = pd.read_csv(songs_file_path)
df_songs


,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,tv_name,song_name_unique,publish_date,is_ost
0,633973719,0029blSZ16tCiJ,纵马踏歌行,《剑来》动画第二季插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,83047554,纵马踏歌行,003otjHh1KR2Lg,176,1768320000,剑来,纵马踏歌行,2026-01-14,1
1,629395245,003cEvPA3QuEqn,荣光,《风过留痕》影视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,81721431,荣光,002CbsMp1BXe73,265,1770084000,风过留痕,荣光,2026-02-03,1
2,370388317,001l38Gx3sl3kP,寻一个你,《苍兰诀》电视剧温情主题曲,刘宇宁,2241311,001Iu4Dv1NzRCD,33455648,苍兰诀 东方幻想影视原声带,001c5tj84NH6dH,267,1660010400,苍兰诀,寻一个你,2022-08-09,1
3,578702351,001QwWqP38bXCs,烽月,电视剧《折腰》情感主题曲/片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,67523527,折腰 影视原声专辑,001NFTqM3zNUv2,265,1747411200,折腰,烽月,2025-05-17,1
4,285903302,004a6xSN489klN,当遇见你,《冰糖炖雪梨》电视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,15738588,当遇见你,0027ZeN63fdE93,191,1583769600,冰糖炖雪梨,当遇见你,2020-03-10,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,448734874,001I5tmS3sA6jB,爱情之所以,《正青春》电视剧插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,43671782,正青春 电视剧原声带,004TS8Lz3FGMHs,207,1723478400,正青春,爱情之所以,2024-08-13,1
146,401100858,0009Ba6M3Rd0kp,望道,《望道》电影同名主题曲,刘宇宁,2241311,001Iu4Dv1NzRCD,36285879,望道,001SGs7l1BP05L,196,1678982400,望道,望道,2023-03-17,1
147,335548330,0032iLRJ3YH8fN,风起时再见,《迷雾追踪》影视剧主题曲,刘宇宁,2241311,001Iu4Dv1NzRCD,33427051,迷雾追踪 影视原声带,003V3TEN24Hj29,260,1607616000,迷雾追踪,风起时再见,2020-12-11,1
148,264776262,002rs2Tk3fYVGh,要替我幸福,《暖暖，请多指教》电视剧片尾曲,刘宇宁,2241311,001Iu4Dv1NzRCD,12373795,暖暖，请多指教 影视原声带,003eJOh54YBBK1,242,1589472000,暖暖，请多指教,要替我幸福,2020-05-15,1


# 分词，词频与词性分析

In [5]:
word_to_fix = {
    '阮': 'r',
    '袂': 'v'
}

In [6]:
def process_lyrics_with_jieba(text):
    # 1. 词性标注与分词
    # jieba.posseg 会同时返回词和词性
    words_with_pos = pseg.cut(text)

    
    # 2. 过滤无意义字符（标点、空格、单字符停用词）
    filtered_data = []
    for word, pos in words_with_pos:
        # 排除标点符号（x表示标点）及空白字符
        if pos != 'x' and len(word.strip()) > 0:
            if word in word_to_fix:
                filtered_data.append((word, word_to_fix[word]))
            else:
                filtered_data.append((word, pos))
    
    # 3. 统计词频
    word_counts = Counter([item[0] for item in filtered_data])
    
    # 4. 汇总信息 (词, 词性, 频数)
    # 我们以词为 Key，存储词性
    word_pos_map = {word: pos for word, pos in filtered_data}
    
    # 排序：按词频从高到低
    sorted_results = []
    for word, count in word_counts.most_common():
        sorted_results.append({
            "word": word,
            "pos": word_pos_map[word],
            "freq": count # 词频
        })
    
    return sorted_results

In [7]:
lyric_file_path = 'data_output/liuyuning/qq_lyric_data_cleared.json'
# 读取歌词文件
with open(lyric_file_path, 'r') as f:
    lyric_data = json.load(f)

In [8]:
lyric_words_dict = {}
for i in lyric_data:
    if i:
        lyric_words_dict[i['song_id']] = process_lyrics_with_jieba(i['lyrics_text'])
lyric_words_dict

Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/zj/y5ymt78d6qd4p13cf1q_3rch0000gn/T/jieba.cache
Loading model cost 0.648 seconds.
Prefix dict has been built successfully.


{633973719: [{'word': '处', 'pos': 'n', 'freq': 4},
  {'word': '了', 'pos': 'ul', 'freq': 4},
  {'word': '把', 'pos': 'p', 'freq': 2},
  {'word': '江湖', 'pos': 'ns', 'freq': 2},
  {'word': '斟入', 'pos': 'v', 'freq': 2},
  {'word': '陈年', 'pos': 'nr', 'freq': 2},
  {'word': '酒', 'pos': 'n', 'freq': 2},
  {'word': '醉眼', 'pos': 'n', 'freq': 2},
  {'word': '望断', 'pos': 'v', 'freq': 2},
  {'word': '十二重', 'pos': 'm', 'freq': 2},
  {'word': '楼', 'pos': 'n', 'freq': 2},
  {'word': '锈剑', 'pos': 'n', 'freq': 2},
  {'word': '挑落', 'pos': 'v', 'freq': 2},
  {'word': '风雪', 'pos': 'n', 'freq': 2},
  {'word': '落款', 'pos': 'n', 'freq': 2},
  {'word': '梅花', 'pos': 'nr', 'freq': 2},
  {'word': '替', 'pos': 'p', 'freq': 2},
  {'word': '旧', 'pos': 'a', 'freq': 2},
  {'word': '恩仇', 'pos': 'nr', 'freq': 2},
  {'word': '长街', 'pos': 'ns', 'freq': 2},
  {'word': '悬满', 'pos': 'v', 'freq': 2},
  {'word': '褪色', 'pos': 'v', 'freq': 2},
  {'word': '灯笼', 'pos': 'n', 'freq': 2},
  {'word': '照见', 'pos': 'v', 'freq': 2},
  {'w

In [9]:
rows = []
for song_id, word_list in lyric_words_dict.items():
    for item in word_list:
        # 创建新字典，保留原始数据并加入歌曲ID列
        new_row = {
            'song_id': song_id,
            'word': item['word'],
            'pos': item['pos'],
            'freq': item['freq']
        }
        rows.append(new_row)

# 3. 转换为 DataFrame
df_word = pd.DataFrame(rows)
df_word

,song_id,word,pos,freq
0,633973719,处,n,4
1,633973719,了,ul,4
2,633973719,把,p,2
3,633973719,江湖,ns,2
4,633973719,斟入,v,2
...,...,...,...,...
13880,297559879,血泪,n,1
13881,297559879,挥洒,v,1
13882,297559879,书写,n,1
13883,297559879,新,a,1


In [10]:
# 合并
# 1. 确保 df_word 的 song_id 是字符串
df_word['song_id'] = df_word['song_id'].astype(str)

# 2. 确保 df_unique 的 song_id 是字符串（并去掉可能存在的空格）
df_songs['song_id'] = df_songs['song_id'].astype(str).str.strip()

# 3. 执行合并
df_merged = df_word.merge(df_songs, on='song_id', how='left')

# 4. 删除空值
df_merged = df_merged.dropna()

df_merged

,song_id,word,pos,freq,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,tv_name,song_name_unique,publish_date,is_ost
0,633973719,处,n,4,0029blSZ16tCiJ,纵马踏歌行,《剑来》动画第二季插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,83047554,纵马踏歌行,003otjHh1KR2Lg,176,1768320000,剑来,纵马踏歌行,2026-01-14,1
1,633973719,了,ul,4,0029blSZ16tCiJ,纵马踏歌行,《剑来》动画第二季插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,83047554,纵马踏歌行,003otjHh1KR2Lg,176,1768320000,剑来,纵马踏歌行,2026-01-14,1
2,633973719,把,p,2,0029blSZ16tCiJ,纵马踏歌行,《剑来》动画第二季插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,83047554,纵马踏歌行,003otjHh1KR2Lg,176,1768320000,剑来,纵马踏歌行,2026-01-14,1
3,633973719,江湖,ns,2,0029blSZ16tCiJ,纵马踏歌行,《剑来》动画第二季插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,83047554,纵马踏歌行,003otjHh1KR2Lg,176,1768320000,剑来,纵马踏歌行,2026-01-14,1
4,633973719,斟入,v,2,0029blSZ16tCiJ,纵马踏歌行,《剑来》动画第二季插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,83047554,纵马踏歌行,003otjHh1KR2Lg,176,1768320000,剑来,纵马踏歌行,2026-01-14,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13880,297559879,血泪,n,1,000sfLHg43UbQo,苍穹之下,电视剧《斗罗大陆》戴沐白人物曲）,刘宇宁,2241311,001Iu4Dv1NzRCD,17500078,斗罗大陆 史兰客七怪音乐专辑,000uo5Ig1624LG,272,1613354400,斗罗大陆,苍穹之下,2021-02-15,1
13881,297559879,挥洒,v,1,000sfLHg43UbQo,苍穹之下,电视剧《斗罗大陆》戴沐白人物曲）,刘宇宁,2241311,001Iu4Dv1NzRCD,17500078,斗罗大陆 史兰客七怪音乐专辑,000uo5Ig1624LG,272,1613354400,斗罗大陆,苍穹之下,2021-02-15,1
13882,297559879,书写,n,1,000sfLHg43UbQo,苍穹之下,电视剧《斗罗大陆》戴沐白人物曲）,刘宇宁,2241311,001Iu4Dv1NzRCD,17500078,斗罗大陆 史兰客七怪音乐专辑,000uo5Ig1624LG,272,1613354400,斗罗大陆,苍穹之下,2021-02-15,1
13883,297559879,新,a,1,000sfLHg43UbQo,苍穹之下,电视剧《斗罗大陆》戴沐白人物曲）,刘宇宁,2241311,001Iu4Dv1NzRCD,17500078,斗罗大陆 史兰客七怪音乐专辑,000uo5Ig1624LG,272,1613354400,斗罗大陆,苍穹之下,2021-02-15,1


In [11]:
words_file_path = 'data_output/liuyuning/qq_words_data.csv'
df_merged.to_csv(words_file_path, index=False)

In [13]:
df_merged[df_merged['word'] == '云']

,song_id,word,pos,freq,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,tv_name,song_name_unique,publish_date,is_ost
1351,615276233,云,n,3,003iHNqk3GEuUU,那朵花,《四喜》电视剧主题曲,刘宇宁,2241311,001Iu4Dv1NzRCD,77827929,四喜 电视剧原声带,000pSMGF4U3SCx,273,1762565400,四喜,那朵花,2025-11-08,1
1994,423942396,云,n,4,001XlHtu40S3Mw,就在江湖之上,《莲花楼》影视剧片头曲,刘宇宁,2241311,001Iu4Dv1NzRCD,40335736,莲花楼 电视剧原声专辑,002CIQEt41OeLt,187,1690106400,莲花楼,就在江湖之上,2023-07-23,1
3750,357646651,云,n,3,003z3gS02JaAd5,凌云寂,《说英雄谁是英雄》网剧插曲,刘宇宁,2241311,001Iu4Dv1NzRCD,28396337,说英雄谁是英雄 网剧原声带,000Eb9LR3t6knO,249,1653616800,说英雄谁是英雄,凌云寂,2022-05-27,1
4432,476897109,云,n,1,004CFtTw0fwohb,世世,《与凤行》电视剧「铭刻入骨」主题曲,刘宇宁,2241311,001Iu4Dv1NzRCD,48449281,与凤行 电视剧原声带,004G90F50JcL55,252,1710727200,与凤行,世世,2024-03-18,1
5458,547010137,云,ns,1,000BLRzY38NExV,年长,《蜀锦人家》影视剧人物情感曲,刘宇宁,2241311,001Iu4Dv1NzRCD,59577263,蜀锦人家 影视原声带,002MgFQM3UJmr4,181,1733191200,蜀锦人家,年长,2024-12-03,1
8008,331063452,云,ns,1,001UXg1r3RmNRf,造化,《魔道祖师》动画魏无羡角色曲,刘宇宁,2241311,001Iu4Dv1NzRCD,23109695,造化,003S7TNB0Y1VKT,282,1635645600,魔道祖师,造化,2021-10-31,1
10864,270355541,云,n,2,000S2jb20DSuC8,浪,《河神2》电视剧主题曲,刘宇宁,2241311,001Iu4Dv1NzRCD,13240340,河神2 网剧原声带,002IbIhh03HaQM,251,1594000800,河神2,浪,2020-07-06,1
